# RoBERTa FOMC Sentiment Classification

Fine-tunes `roberta-base` for Dovish, Hawkish, and Neutral classification. Run the cells from top to bottom.


In [ ]:
"""
COMP9444 FOMC Sentiment Classification

Model:  roberta-base → 3-class classification head → full fine-tuning
Labels: 0 = Dovish, 1 = Hawkish, 2 = Neutral

Outputs per seed
----------------
results/roberta_base_combined/seed_<SEED>/
    config.json
    metrics.json
    predictions.csv
    classification_report.csv
    training_history.csv
    learning_curve.png
    confusion_matrix_raw.csv
    confusion_matrix_normalized.csv
    confusion_matrix_normalized.png

results/roberta_base_combined/
    aggregate_metrics.json

Extension (Combined-S, sentence-split):
results/roberta_base_combined_split/  (same layout)
"""

from __future__ import annotations

import argparse
import csv
import json
import platform
import re
import time
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile
import xml.etree.ElementTree as ET
from collections import Counter

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)


## Paths


In [ ]:
WORKING_DIRECTORY = Path.cwd().resolve()
if (WORKING_DIRECTORY / "9444").is_dir():
    ROOT = WORKING_DIRECTORY / "9444"
elif WORKING_DIRECTORY.name.lower() == "models":
    ROOT = WORKING_DIRECTORY.parent
elif WORKING_DIRECTORY.name == "9444":
    ROOT = WORKING_DIRECTORY
else:
    raise FileNotFoundError(
        "Run this notebook from the project root, 9444, or 9444/models directory."
    )
DATA_ZIP = ROOT / "FOMC_dataset_checkpoint.zip"
RESULTS_ROOT = ROOT / "results"

SEEDS = ["5768", "78516", "944601"]
LABEL_NAMES = ["Dovish", "Hawkish", "Neutral"]
CHECKPOINT = "roberta-base"


## Shared training settings (aligned with BERT / FinBERT members)


In [ ]:
MAX_LENGTH = 256
MAX_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 2
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
DROPOUT = 0.1

# Hyperparameter grid - tune on seed 5768, freeze, run remaining seeds
HP_GRID = [
    {"lr": 1e-5, "batch_size": 16},
    {"lr": 2e-5, "batch_size": 16},
    {"lr": 5e-5, "batch_size": 16},
    {"lr": 1e-5, "batch_size": 8},
    {"lr": 2e-5, "batch_size": 8},
    {"lr": 5e-5, "batch_size": 8},
]


## Data loading (mirrors fomc_dataset_analysis.py, pure-stdlib xlsx reader)


In [ ]:
NS = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"


def _column_index(cell_ref: str) -> int:
    letters = "".join(ch for ch in cell_ref if ch.isalpha())
    idx = 0
    for ch in letters:
        idx = idx * 26 + ord(ch) - ord("A") + 1
    return idx - 1


def _cell_text(cell, shared_strings: list[str]) -> str:
    cell_type = cell.attrib.get("t")
    if cell_type == "inlineStr":
        return "".join(t.text or "" for t in cell.iter(NS + "t"))
    value = cell.find(NS + "v")
    if value is None:
        return ""
    if cell_type == "s":
        return shared_strings[int(value.text)]
    return value.text or ""


def _read_xlsx_bytes(data: bytes) -> list[dict]:
    with ZipFile(BytesIO(data)) as wb:
        shared_strings: list[str] = []
        if "xl/sharedStrings.xml" in wb.namelist():
            root = ET.fromstring(wb.read("xl/sharedStrings.xml"))
            for item in root:
                shared_strings.append(
                    "".join(t.text or "" for t in item.iter(NS + "t"))
                )
        sheet = ET.fromstring(wb.read("xl/worksheets/sheet1.xml"))
        rows: list[list[str]] = []
        for row in sheet.find(NS + "sheetData"):
            values: list[str] = []
            for cell in row:
                idx = _column_index(cell.attrib["r"])
                while len(values) <= idx:
                    values.append("")
                values[idx] = _cell_text(cell, shared_strings)
            rows.append(values)
        headers = rows[0]
        records = []
        for row_vals in rows[1:]:
            rec = {
                headers[i]: row_vals[i] if i < len(row_vals) else ""
                for i in range(len(headers))
            }
            rec["index"] = int(rec["index"])
            rec["year"] = int(rec["year"])
            rec["label"] = int(rec["label"])
            records.append(rec)
    return records


def load_seed(seed: str) -> tuple[list[dict], list[dict]]:
    """Return (train_records, test_records) for a given seed string."""
    with ZipFile(DATA_ZIP) as archive:
        train_path = f"FOMC_dataset_checkpoint/lab-manual-combine-train-{seed}.xlsx"
        test_path = f"FOMC_dataset_checkpoint/lab-manual-combine-test-{seed}.xlsx"
        train_records = _read_xlsx_bytes(archive.read(train_path))
        test_records = _read_xlsx_bytes(archive.read(test_path))
    return train_records, test_records


## Sentence splitting (Extension: Combined-S)


In [ ]:
_SPLIT_PATTERN = re.compile(r"(?<=[.!?])\s+")


def split_into_sentences(text: str) -> list[str]:
    """Naïve sentence splitter - good enough for FOMC prose."""
    parts = _SPLIT_PATTERN.split(text.strip())
    return [p.strip() for p in parts if p.strip()]


def expand_to_sentences(records: list[dict]) -> list[dict]:
    """
    For the Combined-S extension: expand each record into one record per
    sentence, keeping the parent label and a sub-index.
    """
    expanded = []
    for rec in records:
        sentences = split_into_sentences(rec["sentence"])
        if not sentences:
            sentences = [rec["sentence"]]
        for sub_idx, sent in enumerate(sentences):
            new_rec = dict(rec)
            new_rec["sentence"] = sent
            new_rec["parent_index"] = rec["index"]
            new_rec["sub_index"] = sub_idx
            # Assign a unique index
            new_rec["index"] = rec["index"] * 1000 + sub_idx
            expanded.append(new_rec)
    return expanded


## Normalise sentence text (shared pre-processing rule §2.3)


In [ ]:
def normalise(text: str) -> str:
    text = text.strip()
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text


## PyTorch Dataset


In [ ]:
class FOMCDataset(Dataset):
    def __init__(
        self,
        records: list[dict],
        tokenizer,
        max_length: int = MAX_LENGTH,
    ):
        self.records = records
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int):
        rec = self.records[idx]
        encoding = self.tokenizer(
            normalise(rec["sentence"]),
            max_length=self.max_length,
            truncation=True,
            padding=False,
            return_tensors=None,
        )
        return {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
            "label": rec["label"],
            "index": rec["index"],
            "year": rec["year"],
            "sentence": rec["sentence"],
        }


def collate_fn(batch):
    """Dynamic padding within a batch."""
    max_len = max(len(item["input_ids"]) for item in batch)
    input_ids, attention_masks, labels = [], [], []
    meta = {"index": [], "year": [], "sentence": []}

    for item in batch:
        pad_len = max_len - len(item["input_ids"])
        input_ids.append(item["input_ids"] + [1] * pad_len)       # RoBERTa pad_token_id = 1
        attention_masks.append(item["attention_mask"] + [0] * pad_len)
        labels.append(item["label"])
        meta["index"].append(item["index"])
        meta["year"].append(item["year"])
        meta["sentence"].append(item["sentence"])

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "index": meta["index"],
        "year": meta["year"],
        "sentence": meta["sentence"],
    }


## Training helpers


In [ ]:
def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def train_one_epoch(model, loader, optimiser, scheduler, device):
    model.train()
    total_loss = 0.0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimiser.zero_grad()
        output = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = output.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / max(len(loader), 1)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    total_loss = 0.0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        output = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total_loss += output.loss.item()

        probs = torch.softmax(output.logits, dim=-1).cpu().numpy()
        preds = np.argmax(probs, axis=1)
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / max(len(loader), 1)
    wf1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
    return avg_loss, wf1, all_preds, all_labels, all_probs


## Result saving


In [ ]:
def save_config(out_dir: Path, cfg: dict):
    with open(out_dir / "config.json", "w") as f:
        json.dump(cfg, f, indent=2)


def save_metrics(out_dir: Path, metrics: dict):
    with open(out_dir / "metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)


def save_predictions(
    out_dir: Path,
    records: list[dict],
    preds: list[int],
    probs: list[list[float]],
    seed: str,
    model_name: str,
):
    path = out_dir / "predictions.csv"
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "sample_id", "year", "sentence", "true_label",
                "predicted_label", "prob_dovish", "prob_hawkish",
                "prob_neutral", "correct", "seed", "model_name",
            ],
        )
        writer.writeheader()
        for rec, pred, prob in zip(records, preds, probs):
            writer.writerow({
                "sample_id": rec["index"],
                "year": rec["year"],
                "sentence": rec["sentence"],
                "true_label": rec["label"],
                "predicted_label": pred,
                "prob_dovish": f"{prob[0]:.6f}",
                "prob_hawkish": f"{prob[1]:.6f}",
                "prob_neutral": f"{prob[2]:.6f}",
                "correct": int(rec["label"] == pred),
                "seed": seed,
                "model_name": model_name,
            })


def save_classification_report(out_dir: Path, true_labels, pred_labels):
    report = classification_report(
        true_labels, pred_labels,
        target_names=LABEL_NAMES,
        output_dict=True,
        zero_division=0,
    )
    with open(out_dir / "classification_report.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["class", "precision", "recall", "f1-score", "support"])
        for cls in LABEL_NAMES:
            row = report[cls]
            writer.writerow([cls, row["precision"], row["recall"], row["f1-score"], row["support"]])
        for avg_key in ["macro avg", "weighted avg"]:
            row = report[avg_key]
            writer.writerow([avg_key, row["precision"], row["recall"], row["f1-score"], row["support"]])


def save_training_history(out_dir: Path, history: list[dict]):
    if not history:
        return
    fields = list(history[0].keys())
    with open(out_dir / "training_history.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(history)


def save_learning_curve(out_dir: Path, history: list[dict], best_epoch: int):
    epochs = [h["epoch"] for h in history]
    train_loss = [h["train_loss"] for h in history]
    val_loss = [h["val_loss"] for h in history]
    val_wf1 = [h["val_weighted_f1"] for h in history]

    fig, ax1 = plt.subplots(figsize=(8, 5))
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.plot(epochs, train_loss, label="Train Loss", color="tab:blue")
    ax1.plot(epochs, val_loss, label="Val Loss", color="tab:orange")
    ax1.axvline(x=best_epoch, color="gray", linestyle="--", label=f"Best epoch ({best_epoch})")

    ax2 = ax1.twinx()
    ax2.set_ylabel("Validation Weighted F1")
    ax2.plot(epochs, val_wf1, label="Val Weighted F1", color="tab:green")

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower left")

    plt.title("Learning Curve")
    plt.tight_layout()
    fig.savefig(out_dir / "learning_curve.png", dpi=150)
    plt.close(fig)


def save_confusion_matrices(out_dir: Path, true_labels, pred_labels):
    cm = confusion_matrix(true_labels, pred_labels, labels=[0, 1, 2])

    # Raw
    with open(out_dir / "confusion_matrix_raw.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([""] + LABEL_NAMES)
        for i, row in enumerate(cm):
            writer.writerow([LABEL_NAMES[i]] + list(row))

    # Normalised (by true class)
    with np.errstate(divide="ignore", invalid="ignore"):
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        cm_norm = np.nan_to_num(cm_norm)

    with open(out_dir / "confusion_matrix_normalized.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([""] + LABEL_NAMES)
        for i, row in enumerate(cm_norm):
            writer.writerow([LABEL_NAMES[i]] + [f"{v:.4f}" for v in row])

    # Plot
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(LABEL_NAMES)
    ax.set_yticklabels(LABEL_NAMES)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title("Confusion Matrix (Normalised)")
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm_norm[i, j]:.1%}", ha="center", va="center",
                    color="white" if cm_norm[i, j] > 0.5 else "black")
    plt.tight_layout()
    fig.savefig(out_dir / "confusion_matrix_normalized.png", dpi=150)
    plt.close(fig)


## Core training routine for one seed


In [ ]:
def run_seed(
    seed: str,
    out_dir: Path,
    lr: float,
    batch_size: int,
    model_name: str,
    use_sentence_split: bool = False,
    device: torch.device | None = None,
):
    if device is None:
        device = get_device()

    int_seed = int(seed)
    torch.manual_seed(int_seed)
    np.random.seed(int_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int_seed)

    print(f"\n{'=' * 60}")
    print(f"  Seed={seed}  lr={lr}  batch_size={batch_size}  split={use_sentence_split}")
    print(f"{'=' * 60}")

    out_dir.mkdir(parents=True, exist_ok=True)

    # ---- Load data --------------------------------------------------------
    train_raw, test_raw = load_seed(seed)

    if use_sentence_split:
        train_raw = expand_to_sentences(train_raw)
        test_raw = expand_to_sentences(test_raw)

    # ---- Validation split (§2.2) ------------------------------------------
    train_indices, val_indices = train_test_split(
        range(len(train_raw)),
        test_size=0.20,
        random_state=int_seed,
        stratify=[r["label"] for r in train_raw],
    )
    train_records = [train_raw[i] for i in train_indices]
    val_records = [train_raw[i] for i in val_indices]
    test_records = test_raw

    # ---- Tokeniser --------------------------------------------------------
    tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

    train_ds = FOMCDataset(train_records, tokenizer)
    val_ds = FOMCDataset(val_records, tokenizer)
    test_ds = FOMCDataset(test_records, tokenizer)

    _g = torch.Generator()
    _g.manual_seed(int_seed)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              collate_fn=collate_fn, num_workers=0, generator=_g)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            collate_fn=collate_fn, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             collate_fn=collate_fn, num_workers=0)

    # ---- Model ------------------------------------------------------------
    model = AutoModelForSequenceClassification.from_pretrained(
        CHECKPOINT, num_labels=3, hidden_dropout_prob=DROPOUT,
        attention_probs_dropout_prob=DROPOUT,
        ignore_mismatched_sizes=True,
    )
    model.to(device)

    # ---- Optimiser + scheduler -------------------------------------------
    total_steps = len(train_loader) * MAX_EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    optimiser = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = get_linear_schedule_with_warmup(
        optimiser, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )

    # ---- Training loop ----------------------------------------------------
    best_val_wf1 = -1.0
    best_epoch = 1
    patience_counter = 0
    history: list[dict] = []
    best_state = None
    train_start = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, optimiser, scheduler, device)
        val_loss, val_wf1, _, _, _ = evaluate(model, val_loader, device)

        history.append({
            "epoch": epoch,
            "train_loss": round(train_loss, 6),
            "val_loss": round(val_loss, 6),
            "val_weighted_f1": round(val_wf1, 6),
        })
        print(f"  Epoch {epoch:2d} | train_loss={train_loss:.4f}  "
              f"val_loss={val_loss:.4f}  val_wf1={val_wf1:.4f}")

        if val_wf1 > best_val_wf1:
            best_val_wf1 = val_wf1
            best_epoch = epoch
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"  Early stopping at epoch {epoch}.")
                break

    train_time = time.time() - train_start

    # ---- Restore best checkpoint and save to disk -----------------------
    if best_state is not None:
        model.load_state_dict(best_state)
    model.save_pretrained(out_dir / "model")
    tokenizer.save_pretrained(out_dir / "model")

    # ---- Test evaluation -------------------------------------------------
    _, test_wf1, test_preds, test_true, test_probs = evaluate(model, test_loader, device)
    test_mf1 = f1_score(test_true, test_preds, average="macro", zero_division=0)
    test_acc = accuracy_score(test_true, test_preds)

    print(f"  >> Test weighted-F1={test_wf1:.4f}  macro-F1={test_mf1:.4f}  acc={test_acc:.4f}")

    # ---- Save outputs ----------------------------------------------------
    save_training_history(out_dir, history)
    save_learning_curve(out_dir, history, best_epoch)
    save_predictions(out_dir, test_records, test_preds, test_probs, seed, model_name)
    save_classification_report(out_dir, test_true, test_preds)
    save_confusion_matrices(out_dir, test_true, test_preds)

    # Per-class F1
    per_class = f1_score(test_true, test_preds, average=None, zero_division=0).tolist()

    metrics = {
        "seed": seed,
        "model_name": model_name,
        "best_epoch": best_epoch,
        "val_weighted_f1": round(best_val_wf1, 6),
        "test_weighted_f1": round(test_wf1, 6),
        "test_macro_f1": round(test_mf1, 6),
        "test_accuracy": round(test_acc, 6),
        "test_f1_dovish": round(per_class[0], 6),
        "test_f1_hawkish": round(per_class[1], 6),
        "test_f1_neutral": round(per_class[2], 6),
        "train_time_seconds": round(train_time, 2),
        "n_train": len(train_records),
        "n_val": len(val_records),
        "n_test": len(test_records),
    }
    save_metrics(out_dir, metrics)

    # Config
    import transformers, sklearn, torch as _torch
    cfg = {
        "model_name": model_name,
        "pretrained_checkpoint": CHECKPOINT,
        "checkpoint_revision": "main",
        "training_type": "full_fine_tuning",
        "tokenizer": CHECKPOINT,
        "max_length": MAX_LENGTH,
        "learning_rate": lr,
        "batch_size": batch_size,
        "epochs": MAX_EPOCHS,
        "best_epoch": best_epoch,
        "dropout": DROPOUT,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "random_seed": int_seed,
        "train_split_file": f"data/splits/{seed}_train.csv",
        "validation_split_file": f"data/splits/{seed}_val.csv",
        "test_split_file": f"FOMC_dataset_checkpoint/lab-manual-combine-test-{seed}.xlsx",
        "use_sentence_split": use_sentence_split,
        "library_versions": {
            "transformers": transformers.__version__,
            "torch": _torch.__version__,
            "sklearn": sklearn.__version__,
        },
        "hardware": str(device),
        "platform": platform.platform(),
    }
    save_config(out_dir, cfg)

    return metrics


## Hyperparameter search (seed 5768 only)


In [ ]:
def hyperparameter_search(results_dir: Path, use_sentence_split: bool = False) -> tuple[float, int]:
    """Grid-search over HP_GRID on seed 5768. Returns best (lr, batch_size)."""
    device = get_device()
    print("\n=== Hyperparameter search on seed 5768 ===")
    best_lr, best_bs, best_wf1 = HP_GRID[0]["lr"], HP_GRID[0]["batch_size"], -1.0

    for hp in HP_GRID:
        lr, bs = hp["lr"], hp["batch_size"]
        tmp_dir = results_dir / f"_hpsearch_lr{lr}_bs{bs}"
        m = run_seed(
            "5768", tmp_dir, lr=lr, batch_size=bs,
            model_name="roberta_hpsearch",
            use_sentence_split=use_sentence_split,
            device=device,
        )
        if m["val_weighted_f1"] > best_wf1:
            best_wf1 = m["val_weighted_f1"]
            best_lr, best_bs = lr, bs

    print(f"\nBest HP: lr={best_lr}  batch_size={best_bs}  val_wf1={best_wf1:.4f}")
    return best_lr, best_bs


## Aggregate metrics


In [ ]:
def aggregate(seed_metrics: list[dict], out_dir: Path, model_name: str):
    wf1s = [m["test_weighted_f1"] for m in seed_metrics]
    mf1s = [m["test_macro_f1"] for m in seed_metrics]
    accs = [m["test_accuracy"] for m in seed_metrics]

    agg = {
        "model_name": model_name,
        "seeds": SEEDS,
        "test_weighted_f1_mean": round(float(np.mean(wf1s)), 6),
        "test_weighted_f1_std": round(float(np.std(wf1s)), 6),
        "test_macro_f1_mean": round(float(np.mean(mf1s)), 6),
        "test_macro_f1_std": round(float(np.std(mf1s)), 6),
        "test_accuracy_mean": round(float(np.mean(accs)), 6),
        "test_accuracy_std": round(float(np.std(accs)), 6),
        "per_seed": seed_metrics,
    }
    with open(out_dir / "aggregate_metrics.json", "w") as f:
        json.dump(agg, f, indent=2)

    print(f"\n--- Aggregate ({model_name}) ---")
    print(f"  Weighted F1: {agg['test_weighted_f1_mean']:.4f} ± {agg['test_weighted_f1_std']:.4f}")
    print(f"  Macro    F1: {agg['test_macro_f1_mean']:.4f} ± {agg['test_macro_f1_std']:.4f}")
    print(f"  Accuracy:    {agg['test_accuracy_mean']:.4f} ± {agg['test_accuracy_std']:.4f}")


## Main


In [ ]:
def main():
    parser = argparse.ArgumentParser(description="RoBERTa fine-tuning for FOMC sentiment")
    parser.add_argument(
        "--mode",
        choices=["combined", "combined_split", "both"],
        default="combined",
        help="Which experiment to run: combined (unsplit), combined_split (sentence-split), or both",
    )
    parser.add_argument(
        "--skip_hpsearch",
        action="store_true",
        help="Skip hyperparameter search and use defaults (lr=2e-5, batch_size=16)",
    )
    parser.add_argument("--lr", type=float, default=2e-5, help="Override learning rate")
    parser.add_argument("--batch_size", type=int, default=16, help="Override batch size")
    args = parser.parse_args()

    device = get_device()
    print(f"Device: {device}")

    modes = []
    if args.mode in ("combined", "both"):
        modes.append(("combined", False))
    if args.mode in ("combined_split", "both"):
        modes.append(("combined_split", True))

    for exp_name, use_split in modes:
        model_name = f"roberta_base_{exp_name}"
        results_dir = RESULTS_ROOT / f"roberta_base_{exp_name}"
        results_dir.mkdir(parents=True, exist_ok=True)

        # Step 1: hyperparameter search
        if args.skip_hpsearch:
            best_lr, best_bs = args.lr, args.batch_size
            print(f"Skipping HP search. Using lr={best_lr}, batch_size={best_bs}")
        else:
            best_lr, best_bs = hyperparameter_search(results_dir, use_sentence_split=use_split)

        # Step 2: run all three seeds with frozen hyperparameters
        seed_metrics = []
        for seed in SEEDS:
            seed_dir = results_dir / f"seed_{seed}"
            m = run_seed(
                seed, seed_dir,
                lr=best_lr, batch_size=best_bs,
                model_name=model_name,
                use_sentence_split=use_split,
                device=device,
            )
            seed_metrics.append(m)

        # Step 3: aggregate
        aggregate(seed_metrics, results_dir, model_name)


## Experiment configuration

Set the experiment options below. `combined` runs the standard model, while `combined_split` runs the sentence-split extension.


In [ ]:
MODE = "combined"  # "combined", "combined_split", or "both"
SKIP_HPSEARCH = True
LEARNING_RATE = 2e-5
BATCH_SIZE = 16
RUN_SEEDS = SEEDS

print(f"Device: {get_device()}")
print(f"Seeds: {RUN_SEEDS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Batch size: {BATCH_SIZE}")

## Dataset check

Load the first configured split and display its sizes and class distributions before training.


In [ ]:
preview_train, preview_test = load_seed(RUN_SEEDS[0])
print(f"Seed {RUN_SEEDS[0]}")
print(f"Training records: {len(preview_train)}")
print(f"Test records: {len(preview_test)}")
print(f"Training labels: {dict(sorted(Counter(r['label'] for r in preview_train).items()))}")
print(f"Test labels: {dict(sorted(Counter(r['label'] for r in preview_test).items()))}")

## Train and evaluate

This cell logs each epoch, evaluates every configured seed, saves per-seed outputs, and writes aggregate metrics.


In [ ]:
device = get_device()
modes = []
if MODE in ("combined", "both"):
    modes.append(("combined", False))
if MODE in ("combined_split", "both"):
    modes.append(("combined_split", True))

if not modes:
    raise ValueError("MODE must be 'combined', 'combined_split', or 'both'.")

completed_results = {}
for experiment_name, use_sentence_split in modes:
    model_name = f"roberta_base_{experiment_name}"
    results_dir = RESULTS_ROOT / model_name
    results_dir.mkdir(parents=True, exist_ok=True)

    if SKIP_HPSEARCH:
        best_lr, best_batch_size = LEARNING_RATE, BATCH_SIZE
        print(f"Skipping HP search. Using lr={best_lr}, batch_size={best_batch_size}")
    else:
        best_lr, best_batch_size = hyperparameter_search(
            results_dir,
            use_sentence_split=use_sentence_split,
        )

    seed_metrics = []
    for seed in RUN_SEEDS:
        metrics = run_seed(
            seed=seed,
            out_dir=results_dir / f"seed_{seed}",
            lr=best_lr,
            batch_size=best_batch_size,
            model_name=model_name,
            use_sentence_split=use_sentence_split,
            device=device,
        )
        seed_metrics.append(metrics)

    aggregate(seed_metrics, results_dir, model_name)
    completed_results[experiment_name] = results_dir

## Accuracy summary

Read and display the aggregate accuracy and F1 scores saved by the training cell.


In [ ]:
for experiment_name, results_dir in completed_results.items():
    metrics_path = results_dir / "aggregate_metrics.json"
    with metrics_path.open(encoding="utf-8") as file:
        result = json.load(file)

    print(f"\n{result['model_name']}")
    print(f"Accuracy:    {result['test_accuracy_mean']:.4f} ± {result['test_accuracy_std']:.4f}")
    print(f"Weighted F1: {result['test_weighted_f1_mean']:.4f} ± {result['test_weighted_f1_std']:.4f}")
    print(f"Macro F1:    {result['test_macro_f1_mean']:.4f} ± {result['test_macro_f1_std']:.4f}")